# Titanic Survival 🚢

In this notebook, predictive analytics study was performed on a Kaggle Competition data set which consists of the passenger survival data of Titanic disaster.

The breakdown of the study is as below:

<ul>
<li>Data exploration</li>
<li>Data transformation</li>
<li>Feature selection</li>
<li>Model construction</li>
</ul>

Competition Score: 0.78

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as numpy
import pandas as pandas
import matplotlib.pyplot as pyplot
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

# Data Exploration

<ul>
<li>Extract a sample from data set</li>
<li>Get information about data set</li>
<li>Count the number of unique values for each column</li>
<li>Classify variables with the help of viewed sample, data type information and number of unique values</li>
<li>Investigate the distribution of target variable</li>
</ul>

In [ ]:
TrainData = pandas.read_csv('/kaggle/input/titanic/train.csv')
TestData = pandas.read_csv('/kaggle/input/titanic/test.csv')
TrainData.sample(10)

In [ ]:
TrainData.info()

In [ ]:
TrainData.nunique()

In [ ]:
TargetVariable = 'Survived'
CategoricalVariables = ['Pclass', 'Sex', 'Embarked', 'SibSp', 'Parch']
ContinuousVariables = ['Age', 'Fare']
RedundantVariables = ['PassengerId', 'Name', 'Ticket', 'Cabin']

In [ ]:
TrainData[TargetVariable].value_counts().plot(kind='pie', title='Target Variable: Survived')

# Data Transformation

<ul>
<li>Fill missing values with the mode value of each categorical column in both training and test data set</li>
<li>Fill missing values with the mean value of each continuous column in both training and test data set</li>
<li>Map the nominal values of target variable with integer values to construct model properly</li>
<li>Extract a sample from transformed data set</li>
<li>Get information about transformed data set</li>
</ul>

In [ ]:
TrainData['Age'].fillna(TrainData['Age'].mean(), inplace=True)
TrainData['Embarked'].fillna(TrainData['Embarked'].mode()[0], inplace=True)
TrainData['Sex'] = TrainData['Sex'].map({'female':0, 'male':1})
TrainData['Embarked'] = TrainData['Embarked'].map({'C':1, 'S':2, 'Q':3})
TrainData = TrainData.drop(RedundantVariables, axis=1)

TestData['Age'].fillna(TestData['Age'].median(), inplace=True)
TestData['Fare'].fillna(TestData['Fare'].mean(), inplace=True)
TestData['Embarked'].fillna(TestData['Embarked'].mode()[0], inplace=True)
TestData['Sex'] = TestData['Sex'].map({'female':0, 'male':1})
TestData['Embarked'] = TestData['Embarked'].map({'C':1, 'S':2, 'Q':3})

TrainData.sample(10)

In [ ]:
TrainData.info()

# Feature Selection

Target variable of this study is categorical and there is both categorical and continuous variables among candidate features. During feature selection step, we have to seek for a variation of the pattern between target variable and candidate features. Visualization methods can help us to make investigation however the relationship between variables should be statistically proven.

<ul>
<li> Investigate the relationship between target variable and categorical variables
    <ul>
    <li>Make cross tab to count target variable for each categorical variable</li>
    <li>Plot the cross tab values as a bar chart for visual evaluation</li>
    <li>Perform Chi Square Test for statistical evaluation</li>
    </ul>
</li>
<li> Investigate the relationship between target variable and continuous variables
    <ul>
    <li>Plot a Boxplot of each variable for visual evaluation</li>
    <li>Perform ANOVA Test for statistical evaluation</li>
    </ul>
</li>
</ul>

In [ ]:
for variable in CategoricalVariables:
    CrossTabDataFrame = pandas.crosstab(index=TrainData[variable], columns=TrainData[TargetVariable])
    CrossTabDataFrame.plot.bar(title=variable)
    ChiSquareTest = chi2_contingency(CrossTabDataFrame)
    if (ChiSquareTest[1] < 0.05):
        print(variable, "is related with target variable\nP Value:", ChiSquareTest[1])
    else:
        print(variable, "is NOT related with target variable\nP Value:", ChiSquareTest[1])


In [ ]:
for variable in ContinuousVariables:
    TrainData.boxplot(column=variable, by=TargetVariable)
    AnovaTest = f_oneway(TrainData[TargetVariable], TrainData[variable])
    if (AnovaTest[1] < 0.05):
        print(variable, "is related with target variable\nP Value:", AnovaTest[1])
    else:
        print(variable, "is NOT related with target variable\nP Value:", AnovaTest[1])


# Decision Tree

Decision Tree Classification is an appropriate algorithm to solve this problem.

<ul>
<li>Set predictor variables after feature selection</li>
<li>Set train and test vectors</li>
<li>Create and fit a decision tree</li>
<li>Make predictions for submission</li>
<li>Visualize decision tree</li>
</ul>

Competition Score: %78 Accuracy

In [ ]:
PredictorVariables = ['Pclass', 'Sex', 'Embarked', 'SibSp', 'Parch', 'Age', 'Fare']

X_train = TrainData[PredictorVariables]
Y_train = TrainData[TargetVariable]
X_test = TestData[PredictorVariables]

DecisionTree = DecisionTreeClassifier(criterion="entropy", max_depth=4)
DecisionTree.fit(X_train, Y_train)

Y_pred = DecisionTree.predict(X_test)

pyplot.figure(figsize=(15, 10))

plot_tree(DecisionTree)